# Transformer Training

The VQ-VAE encoder is run once over the full dataset to produce token sequences, which are then stored and used to train the transformer. This avoids re-encoding images every epoch.

In [6]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm 

%run 01_vqvae.ipynb
%run 03_transformer.ipynb

input:  torch.Size([4, 3, 32, 32])
x_hat: torch.Size([4, 3, 32, 32])
z_e: torch.Size([4, 32, 8, 8])
z_q: torch.Size([4, 32, 8, 8])
idx: torch.Size([4, 8, 8])
using mps
torch.Size([4, 64, 512])


## Encode Dataset

Run the frozen VQ-VAE encoder once over all of CIFAR-10, saving token sequences of shape `(50000, 65)` — class label followed by 64 codebook indices (8×8 grid).

In [7]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

vqvae = VQVAE(K=512, d=32).to(device)
vqvae.load_state_dict(torch.load('../data/vqvae_cifar10.pt', map_location=device))
vqvae.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
dataset    = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False)

all_seqs = []
with torch.no_grad():
    for x, labels in tqdm(dataloader):
        x      = x.to(device)
        _, _, _, idx = vqvae(x)          # idx: (B, 8, 8)
        idx    = idx.view(x.shape[0], -1).cpu()  # (B, 49)
        labels = labels.unsqueeze(1)     # (B, 1)
        seq    = torch.cat([labels, idx], dim=1)  # (B, 50)
        all_seqs.append(seq)

all_seqs = torch.cat(all_seqs, dim=0)   # (50000, 65)
torch.save(all_seqs, '../data/cifar10_tokens.pt')
print(f'saved: {all_seqs.shape}')

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 196/196 [00:03<00:00, 60.88it/s]

saved: torch.Size([50000, 65])


## Training

In [8]:
all_seqs   = torch.load('../data/cifar10_tokens.pt')
token_dataset  = torch.utils.data.TensorDataset(all_seqs)
token_loader   = torch.utils.data.DataLoader(token_dataset, batch_size=128, shuffle=True)

In [ ]:
gpt       = GPT(K=512, S=8, d_model=256, nhead=8, num_layers=8).to(device)
optimizer = torch.optim.Adam(gpt.parameters(), lr=3e-4)
loss_fn   = nn.CrossEntropyLoss()

epochs = 100
losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    pbar = tqdm(token_loader, desc=f'epoch {epoch+1}/{epochs}', leave=False)
    for (seq,) in pbar:
        seq    = seq.to(device)          # (B, 50)
        inp    = seq[:, :-1]             # (B, 49) — input: label + first 48 tokens
        target = seq[:, 1:]             # (B, 49) — target: first 49 tokens

        logits = gpt(inp)               # (B, 49, K)
        # CrossEntropyLoss expects (B, K, S)
        loss   = loss_fn(logits.transpose(1, 2), target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg = epoch_loss / len(token_loader)
    losses.append(avg)
    print(f'epoch {epoch+1:2d}  loss {avg:.4f}')

plt.plot(losses)
plt.xlabel('epoch')
plt.ylabel('cross-entropy loss')
plt.show()

epoch  1  loss 4.7652


epoch  2  loss 4.2286


epoch  3  loss 4.1299


epoch  4  loss 4.0772


epoch  5  loss 4.0395


...

epoch 86  loss 3.1814


epoch 87  loss 3.1781


epoch 88  loss 3.1751


epoch 89  loss 3.1715


epoch 90/100:  81%|███████████████████████████████████████████████████████████████████████████████████████████████████████████▏                         | 315/391 [01:35<00:22,  3.36it/s, loss=3.2101]

In [ ]:
torch.save(gpt.cpu().state_dict(), '../data/gpt_cifar10_large.pt')
gpt.to(device)